In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")


CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce GTX 1660 Ti


In [2]:
#!/usr/bin/env python3

from ultralytics import YOLO

model = YOLO("yolo12n.pt")  # works out-of-the-box

results = model.train(
    data="/home/fildo7525/SDU/MasterThesis/AI/yolo12/combined_dataset_nomalised/sorted/dataset.yaml",
    epochs=60,        # adjust depending on dataset size
    imgsz=1024,        # big enough for small plants
    batch=4,
    device=0,
    patience=30,       # early stopping
    freeze=0,          # fine-tune everything
    mosaic=0,          # turn off mosaic
    mixup=0,           # turn off mixup
    hsv_h=0, hsv_s=0, hsv_v=0,   # no color augmentation
    degrees=0, scale=0, translate=0, shear=0, perspective=0,
    val=True   # <-- ensures validation is computed
)

print("Training complete.")


WARNING ⚠️ Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at '/home/fildo7525/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.0 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 5740MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/fildo7525/SDU/MasterThesis/AI/yolo12/combined_dataset_nomalised/sorted/dataset.yaml, degrees=0, 

In [1]:
# test on the test set
import os
from pathlib import Path
from unittest import result
from ultralytics import YOLO

test_image = "/home/fildo7525/SDU/MasterThesis/AI/yolo12/combined_dataset_nomalised/sorted/images/test"
images = os.listdir(test_image)
num_images_to_check = 10

model = YOLO("/home/fildo7525/SDU/MasterThesis/AI/yolo12/combined_dataset_nomalised/sorted/yolo12n_finetuned.pt")

run: int = 0
prediction_output = Path(test_image).parent.parent / "predictions"

if prediction_output.exists():
    run = len(list(prediction_output.glob("run*")))

prediction_output = prediction_output / f"run{run}"
while prediction_output.exists():
    run += 1
    prediction_output = prediction_output.parent / f"run{run}"
prediction_output.mkdir(parents=True, exist_ok=True)

for img_name in images:
    img_path = os.path.join(test_image, img_name)
    results = model.predict(source=img_path, conf=0.25, save=True)

    prediction_file = img_name.replace(".jpg", ".txt")

    for result in results:
        if not result.boxes:
            continue

        for box in result.boxes:
            cls = int(box.cls[0])
            conf = box.conf[0]
            xyxy = box.xyxy[0].tolist()
            with open (prediction_output / prediction_file, "a") as f:
                f.write(f"{cls} {conf:.4f} {' '.join(f'{coord:.2f}' for coord in xyxy)}\n")

    print(f"Inference complete for {img_name}. Results saved.")


# results = model.predict(source=test_image, conf=0.25, save=True)
print("Inference complete. Results saved.")


image 1/1 /home/fildo7525/SDU/MasterThesis/AI/yolo12/combined_dataset_nomalised/sorted/images/test/Bjørnkjærvej_TestFlight_2_bigger_yolo12_tile_39_3_NEN_png.rf.a6798256d30cfdd5e7c4df2c430e5429.jpg: 1024x1024 1 Potatoes, 21.5ms
Speed: 6.4ms preprocess, 21.5ms inference, 24.5ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /home/fildo7525/SDU/MasterThesis/runs/detect/predict15
Inference complete for Bjørnkjærvej_TestFlight_2_bigger_yolo12_tile_39_3_NEN_png.rf.a6798256d30cfdd5e7c4df2c430e5429.jpg. Results saved.

image 1/1 /home/fildo7525/SDU/MasterThesis/AI/yolo12/combined_dataset_nomalised/sorted/images/test/Bjørnkjærvej_TestFlight_2_bigger_yolo12_tile_13_3_NEN_png.rf.02e7a5adfac5bc31c8d91a67181a27d2.jpg: 1024x1024 (no detections), 21.3ms
Speed: 5.4ms preprocess, 21.3ms inference, 1.1ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /home/fildo7525/SDU/MasterThesis/runs/detect/predict15
Inference complete for Bjørnkjærvej_TestFlight_2_bigger_yol

In [ ]:
# save model
model.save("/home/fildo7525/SDU/MasterThesis/AI/yolo12/combined_dataset_nomalised/sorted/yolo12n_finetuned.pt")